# 01_08 — Filter & preprocess PacBio data

Load the all-samples **recollapsed** PacBio objects at the gene / pbids / ensemblids / ensemblids_annotatedonly levels and produce filtered, cell-type-labeled objects:


In [4]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import os, gc
import numpy as np
import pandas as pd
import scanpy as sc
gc.enable()

BASEDIR = "./../pacbio"
H5AD_DIR = f"{BASEDIR}/h5ads/"

LEVELS = ["genes", "pbids", "ensemblids", "ensemblids_annotatedonly"]
# LEVELS = [ "pbids", "ensemblids", "ensemblids_annotatedonly"]

# reload figure_paths: a live kernel caches FIG_MAP
import importlib
import figure_paths
importlib.reload(figure_paths)
from figure_paths import figpath  # routes figures/ -> figures/figureN/ via FIG_MAP


In [5]:
# --- PopV predictions + cell-ontology / broad-class maps (same sources as nb 03_01) ---
celltype_df = pd.read_csv("./../csvs/all_samples_recollapsed_popv_predictions_df.csv", index_col=0)
print(celltype_df.shape)
popv_to_ts = pd.read_csv("./../csvs/popv_to_ts_celltype_mapping.csv")
ts_celltypes = pd.read_csv("./../csvs/ts_celltypes.csv")

popv_to_ts_map = dict(zip(popv_to_ts["popv_prediction"], popv_to_ts["ts_cell_ontology_class"]))
ts_broad_map = dict(zip(ts_celltypes["cell_ontology_class"], ts_celltypes["broad_cell_class"]))
print(f"PopV predictions: {celltype_df.shape[0]:,} cells; columns: {list(celltype_df.columns)}")

(293501, 12)
PopV predictions: 293,501 cells; columns: ['popv_knn_on_scvi_prediction', 'popv_scanvi_prediction', 'popv_svm_prediction', 'popv_xgboost_prediction', 'popv_onclass_prediction', 'popv_celltypist_prediction', 'popv_prediction', 'popv_prediction_score', 'popv_majority_vote_prediction', 'popv_majority_vote_score', 'popv_parent', 'tissue']


In [6]:
MIN_UMI_PER_CELL = 1000      # keep cells with total UMI > this
MIN_CELLS_PER_GENE = 5       # keep genes detected in >= this many cells (drops their pbids/ensemblids/ensemblids_annotatedonly too)
MIN_CELLS_PER_MOLECULE = 5   # finer levels: also keep individual pbids/ensemblids/ensemblids_annotatedonly detected in >= this many cells
MIN_MOLECULE_COUNTS = 10     # finer levels: also keep features whose TOTAL count across all cells is >= this
MIN_SAMPLES_DETECTED = 1     # finer levels: also keep features detected (>0) in >= this many samples (tube_id)

In [7]:
# --- Cross-run barcode-collision (contamination) filter (defined here so preprocess() can
# --- call it as its FIRST step, before any QC). Drops 10X barcodes shared between two 10X
# --- runs (LR_library_id) of the same donor (ambient / index hopping).
def filter_barcode_collisions(data, group_field):
    """Iteratively drop 10X barcodes shared between any two runs (group_field) within `data`.
    Returns (overlap_dict, surviving_data). Order-deterministic given `data`."""
    unique_groups = data[group_field].unique()
    overlap_dict = {}
    for i in range(len(unique_groups)):
        for j in range(i + 1, len(unique_groups)):
            gi = data[data[group_field] == unique_groups[i]]
            gj = data[data[group_field] == unique_groups[j]]
            inter = np.intersect1d(gi["10X_barcode"], gj["10X_barcode"])
            denom = gi.shape[0] + gj.shape[0] - len(inter)
            pct = (len(inter) / denom * 100) if denom else 0.0
            overlap_dict[(unique_groups[i], unique_groups[j])] = pct
            overlap_dict[(unique_groups[j], unique_groups[i])] = pct
            data = data[~data["10X_barcode"].isin(inter)]
    return overlap_dict, data

def contamination_keep_mask(obs):
    """Boolean array over obs.index: True = keep (no cross-run barcode collision within its donor)."""
    o = obs[["donor", "LR_library_id"]].copy()
    o["10X_barcode"] = [str(x).split("-1")[0] for x in obs.index]   # PacBio-orientation barcode
    kept_parts = []
    for donor in o["donor"].unique():
        _, dd = filter_barcode_collisions(o[o["donor"] == donor], "LR_library_id")
        kept_parts.append(dd)
    kept_idx = pd.concat(kept_parts).index if kept_parts else obs.index[:0]
    return obs.index.isin(kept_idx)


def add_labels(adata):
    """Attach PopV prediction + score, cell_ontology_class, broad_cell_class to obs."""
    ct = celltype_df.reindex(adata.obs_names)
    adata.obs["popv_prediction"] = ct["popv_prediction"].fillna("unknown").values
    adata.obs["popv_prediction_score"] = pd.to_numeric(ct["popv_prediction_score"], errors="coerce").values
    adata.obs["cell_ontology_class"] = adata.obs["popv_prediction"].map(popv_to_ts_map).fillna("unknown")
    adata.obs["broad_cell_class"] = adata.obs["cell_ontology_class"].map(ts_broad_map).fillna("unknown")
    return adata

def preprocess(level, save=True, keep_cells=None, keep_genes=None):
    in_path = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_{level}_bc_anndata_mincounts500.h5ad"
    adata = sc.read_h5ad(in_path)
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    n_cells0, n_feat0 = adata.shape

    # --- cell filter: CONTAMINATION FIRST, then the UMI QC cut ---
    # At the gene level we (0) drop contamination cells (cross-run 10X-barcode collisions within
    # a donor) on the RAW cells BEFORE any QC, then (1) apply the > MIN_UMI_PER_CELL library-size
    # cut on the full gene matrix. The finer levels instead keep exactly the gene-level cell set
    # (keep_cells), so they inherit both the decontamination and the UMI cut. Running
    # contamination first means every QC filter (UMI, gene detection, per-feature) is computed on
    # the already-decontaminated cells.
    if keep_cells is not None:
        # keep exactly the cells retained at the gene level (already decontaminated + QC'd),
        # regardless of this level's own UMI total
        adata = adata[adata.obs_names.isin(keep_cells)].copy()
        cell_filter = "match genes"
        contam_removed = None
    else:
        # 0. contamination filter FIRST (cross-run barcode collisions within a donor)
        n_precontam = adata.n_obs
        adata = adata[contamination_keep_mask(adata.obs)].copy()
        contam_removed = n_precontam - adata.n_obs
        # 1. QC cell filter: > MIN_UMI_PER_CELL total UMIs (on the decontaminated full gene matrix)
        total = np.asarray(adata.X.sum(axis=1)).ravel()
        adata = adata[total >= MIN_UMI_PER_CELL].copy()
        cell_filter = f"contam -{contam_removed:,}, then > {MIN_UMI_PER_CELL} UMI"

    # --- gene filter (AFTER the cell filter, on the surviving cells) ---
    # At the gene level, keep genes detected in >= MIN_CELLS_PER_GENE of the retained cells
    # and record the surviving gene symbols; at the finer levels, drop every
    # pbids/ensemblids/ensemblids_annotatedonly whose gene was dropped (matched by the shared `gene_name`
    # symbol column).
    computed_keep_genes = None
    if level == "genes":
        cells_per_gene = np.asarray((adata.X > 0).sum(axis=0)).ravel()
        adata = adata[:, cells_per_gene >= MIN_CELLS_PER_GENE].copy()
        computed_keep_genes = set(adata.var["gene_name"].astype(str))
    elif keep_genes is not None:
        adata = adata[:, adata.var["gene_name"].astype(str).isin(keep_genes)].copy()
    n_feat_gene = adata.n_vars

    # --- molecule/feature filters (AFTER the gene filter, on the surviving cells) ---
    # For the finer levels, apply THREE per-feature filters in sequence, tracking the feature
    # count after each so the summary shows how many each threshold removes:
    #   1. MIN_CELLS_PER_MOLECULE : detected (>0) in >= this many retained cells.
    #   2. MIN_MOLECULE_COUNTS    : TOTAL count across all cells >= this.
    #   3. MIN_SAMPLES_DETECTED   : detected (>0) in >= this many distinct samples (tube_id).
    # Runs for every non-"genes" level (pbids, ensemblids, ensemblids_annotatedonly); the gene level is
    # skipped (its features are genes, already thresholded by the gene filter above).
    n_feat_cellcut = None    # after filter 1 (>= MIN_CELLS_PER_MOLECULE cells)
    n_feat_mincount = None   # after filter 2 (+ >= MIN_MOLECULE_COUNTS total counts)
    if level != "genes":
        # 1. per-feature min-cells cut (shrinks the matrix before the heavier steps)
        cells_per_feat = np.asarray((adata.X > 0).sum(axis=0)).ravel()
        adata = adata[:, cells_per_feat >= MIN_CELLS_PER_MOLECULE].copy()
        n_feat_cellcut = adata.n_vars

        # 2. min TOTAL counts across all cells
        feat_total = np.asarray(adata.X.sum(axis=0)).ravel()
        adata = adata[:, feat_total >= MIN_MOLECULE_COUNTS].copy()
        n_feat_mincount = adata.n_vars

        # 3. min number of distinct samples (tube_id) the feature is detected in
        Xcsr = adata.X.tocsr()
        tube = adata.obs["tube_id"].astype(str).values
        n_samples_detected = np.zeros(adata.n_vars, dtype=np.int32)
        for t in np.unique(tube):
            rows = np.where(tube == t)[0]
            n_samples_detected += (np.asarray((Xcsr[rows] > 0).sum(axis=0)).ravel() > 0)
        adata = adata[:, n_samples_detected >= MIN_SAMPLES_DETECTED].copy()
    n_feat_mol = adata.n_vars   # final feature count after all applicable filters

    # labels
    add_labels(adata)

    # warn about (non-unknown) PopV labels missing from popv_to_ts_celltype_mapping.csv
    valid = ~adata.obs["popv_prediction"].astype(str).str.lower().isin(["unknown", "nan", ""])
    unmapped = adata.obs.loc[valid & ~adata.obs["popv_prediction"].isin(popv_to_ts_map),
                             "popv_prediction"].value_counts()
    if len(unmapped):
        print(f"  [{level}] {int(unmapped.sum()):,} cells have a PopV label NOT in "
              f"popv_to_ts_celltype_mapping.csv (-> cell_ontology_class 'unknown'):")
        for lab, n in unmapped.items():
            print(f"      {lab!r}: {n:,} cells")
    else:
        print(f"  [{level}] all PopV labels are present in popv_to_ts_celltype_mapping.csv")

    # drop cells with NA / Unknown PopV prediction — only for the independently-filtered
    # gene level; when matching the gene level, the cell set is already cleaned there.
    if keep_cells is None:
        adata = adata[valid.values].copy()

    # per-cell library size (total_counts) on this level's final feature set, stored in obs
    total_counts = np.asarray(adata.X.sum(axis=1)).ravel()
    adata.obs["total_counts"] = total_counts

    # recompute QC metrics on the FINAL filtered object so var stats (n_cells_by_counts,
    # mean_counts, pct_dropout_by_counts, ...) and obs stats reflect the current cell/feature
    # set rather than stale values carried from the mincounts500 input.
    sc.pp.calculate_qc_metrics(adata, inplace=True)

    out_path = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_{level}_bc_anndata_preprocessed.h5ad"
    if save:
        adata.write_h5ad(out_path)

    # per-threshold removal counts (finer levels only; None at the gene level)
    removed_by_mincounts = (n_feat_cellcut - n_feat_mincount) if n_feat_mincount is not None else None
    removed_by_minsamples = (n_feat_mincount - n_feat_mol) if n_feat_mincount is not None else None

    rec = {
        "level": level,
        "features_in": n_feat0,
        "features_after_gene_filter": n_feat_gene,
        "features_after_mincells": n_feat_cellcut,      # >= MIN_CELLS_PER_MOLECULE cells (finer levels)
        "features_after_mincounts": n_feat_mincount,    # + total counts >= MIN_MOLECULE_COUNTS
        "features_after_minsamples": n_feat_mol,        # + detected in >= MIN_SAMPLES_DETECTED samples (final)
        "removed_by_mincounts": removed_by_mincounts,
        "removed_by_minsamples": removed_by_minsamples,
        "contam_removed": contam_removed,               # cross-run barcode collisions dropped (gene level; None on inherited levels)
        "cells_in": n_cells0,
        "cells_out": adata.n_obs,
        "cell_filter": cell_filter,
        "n_features": adata.n_vars,
        "min_total_counts": float(total_counts.min()) if adata.n_obs else float("nan"),
        "median_popv_score": float(np.nanmedian(adata.obs["popv_prediction_score"])),
        "out_path": os.path.basename(out_path),
    }
    if level == "genes":
        print(f"[{level}] features {n_feat0:,} -> {n_feat_gene:,} (>= {MIN_CELLS_PER_GENE}-cell genes) "
              f"| cells {n_cells0:,} -> {adata.n_obs:,} ({cell_filter}) "
              f"| min total_counts {total_counts.min() if adata.n_obs else float('nan'):,.0f} | saved {out_path}")
    else:
        print(f"[{level}] features {n_feat0:,} -> {n_feat_gene:,} (gene) "
              f"-> {n_feat_cellcut:,} (>= {MIN_CELLS_PER_MOLECULE} cells) "
              f"-> {n_feat_mincount:,} (>= {MIN_MOLECULE_COUNTS} counts, -{removed_by_mincounts:,}) "
              f"-> {n_feat_mol:,} (>= {MIN_SAMPLES_DETECTED} samples, -{removed_by_minsamples:,}) "
              f"| cells {n_cells0:,} -> {adata.n_obs:,} ({cell_filter}) "
              f"| min total_counts {total_counts.min() if adata.n_obs else float('nan'):,.0f} | saved {out_path}")
    kept = list(adata.obs_names)
    del adata
    gc.collect()
    return rec, kept, computed_keep_genes

In [8]:
# Process levels in order. Genes go first: the gene level is the ONLY level that applies the
# > MIN_UMI_PER_CELL cell filter (and drops NA/Unknown-PopV cells). Its surviving cell set is
# then reused verbatim for pbids/ensemblids/ensemblids_annotatedonly (keep_cells=gene_cells), so every
# level keeps exactly the cells that pass at the gene level instead of applying its own
# (lower-UMI) cut. keep_genes still drops the finer features whose gene was removed by the
# gene-level >= MIN_CELLS_PER_GENE filter.
records = []
keep_genes = None
gene_cells = None
kept_by_level = {}
for level in LEVELS:
    kc = None if level == "genes" else gene_cells
    rec, kept, kg = preprocess(level, keep_cells=kc, keep_genes=keep_genes)
    records.append(rec)
    kept_by_level[level] = kept
    if level == "genes":
        keep_genes = kg
        gene_cells = set(kept)

# Harmonize cells across all four objects so they share an IDENTICAL cell set: intersect every
# level's kept cells, then subset (and re-save) any object that still has extra cells. In
# practice only the gene object differs — it was filtered independently and can retain a few
# cells absent from the finer objects (which carry their own 01_03 min-500 barcode set), so this
# subsets the gene object down to match pbids/ensemblids/ensemblids_annotatedonly.
common_cells = set.intersection(*[set(k) for k in kept_by_level.values()])
rec_by_level = {r["level"]: r for r in records}
for level in LEVELS:
    if len(kept_by_level[level]) != len(common_cells):
        path = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_{level}_bc_anndata_preprocessed.h5ad"
        a = sc.read_h5ad(path)
        a = a[a.obs_names.isin(common_cells)].copy()
        a.write_h5ad(path)
        rec_by_level[level]["cells_out"] = a.n_obs
        rec_by_level[level]["min_total_counts"] = float(a.obs["total_counts"].min()) if a.n_obs else float("nan")
        print(f"[{level}] harmonized to common cell set -> {a.n_obs:,} cells | re-saved {path}")
        del a
        gc.collect()

print(f"\nCommon cell set across all levels: {len(common_cells):,} cells")
summary = pd.DataFrame(records)
summary


  [genes] all PopV labels are present in popv_to_ts_celltype_mapping.csv
[genes] features 69,580 -> 49,124 (>= 5-cell genes) | cells 293,501 -> 203,311 (contam -3,950, then > 1000 UMI) | min total_counts 998 | saved ./../pacbio/h5ads//all_samples_pacbio_recollapsed_raw_counts_genes_bc_anndata_preprocessed.h5ad
  [pbids] all PopV labels are present in popv_to_ts_celltype_mapping.csv
[pbids] features 9,504,172 -> 9,460,523 (gene) -> 1,334,005 (>= 5 cells) -> 854,410 (>= 10 counts, -479,595) -> 854,410 (>= 1 samples, -0) | cells 293,501 -> 203,311 (match genes) | min total_counts 305 | saved ./../pacbio/h5ads//all_samples_pacbio_recollapsed_raw_counts_pbids_bc_anndata_preprocessed.h5ad
  [ensemblids] all PopV labels are present in popv_to_ts_celltype_mapping.csv
[ensemblids] features 8,256,983 -> 8,214,493 (gene) -> 810,322 (>= 5 cells) -> 489,842 (>= 10 counts, -320,480) -> 489,842 (>= 1 samples, -0) | cells 293,501 -> 203,311 (match genes) | min total_counts 305 | saved ./../pacbio/h5ad

,level,features_in,features_after_gene_filter,features_after_mincells,features_after_mincounts,features_after_minsamples,removed_by_mincounts,removed_by_minsamples,contam_removed,cells_in,cells_out,cell_filter,n_features,min_total_counts,median_popv_score,out_path
0,genes,69580,49124,NaN,NaN,49124,NaN,NaN,3950.0,293501,203311,"contam -3,950, then > 1000 UMI",49124,998.0,5.0,all_samples_pacbio_recollapsed_raw_counts_gene...
1,pbids,9504172,9460523,1334005.0,854410.0,854410,479595.0,0.0,NaN,293501,203311,match genes,854410,305.0,5.0,all_samples_pacbio_recollapsed_raw_counts_pbid...
2,ensemblids,8256983,8214493,810322.0,489842.0,489842,320480.0,0.0,NaN,293501,203311,match genes,489842,305.0,5.0,all_samples_pacbio_recollapsed_raw_counts_ense...
3,ensemblids_annotatedonly,202807,198294,162279.0,145051.0,145051,17228.0,0.0,NaN,293501,203311,match genes,145051,271.0,5.0,all_samples_pacbio_recollapsed_raw_counts_ense...


## Contamination filter — applied FIRST (inside `preprocess`)

The cross-run 10X-barcode collision filter (drop barcodes shared between two 10X runs / `LR_library_id` of the same **donor** — ambient / index hopping) now runs as the **first** step of `preprocess()` at the gene level, **before** the UMI / gene / per-feature QC filters. The finer levels inherit the decontaminated cell set via `keep_cells=gene_cells`, so all four objects stay on one identical, decontaminated cell set. The cell below just **reports** how many cells were removed — it no longer re-reads or re-writes the saved objects.

In [18]:
# Contamination filter now runs FIRST, inside preprocess() at the gene level (see above), and
# the finer levels inherit the decontaminated cell set via keep_cells=gene_cells. Nothing to
# re-apply here — this cell just reports how many cells the contamination filter removed
# (shared across all levels, since all four share one cell set).
gene_rec = next(r for r in records if r["level"] == "genes")
print(f"Contamination filter (applied FIRST, gene level): removed "
      f"{gene_rec['contam_removed']:,} cells before QC.")
print(f"Final common cell set across all levels: {len(common_cells):,} cells.")
contam_summary = pd.DataFrame(
    [{"level": r["level"], "contam_removed": r.get("contam_removed"), "cells_out": r["cells_out"]}
     for r in records]
)
contam_summary

Contamination filter (applied FIRST, gene level): removed 3,950 cells before QC.
Final common cell set across all levels: 203,311 cells.


,level,contam_removed,cells_out
0,genes,3950.0,203311
1,molecules,NaN,203311
2,transcripts,NaN,203311
3,isoforms,NaN,203311


## Contamination filter — barcode-collision visualizations

Diagnostics for the cross-run 10X-barcode-collision (contamination) filter applied **FIRST** inside `preprocess()` above. These reconstruct the same per-donor / all-donor overlap structure from the raw gene-level object and visualize it:

- **Per-donor overlap heatmaps** — % barcode overlap between each pair of 10X runs (`LR_library_id`) within a donor.
- **All-runs overlap heatmap** — the same across every run, all donors pooled.
- **Per-run scatter / bar** — number of dropped (colliding) barcodes per run vs its total cell count.

Figures saved as `figures/01_08_*`. The dropped-cell count reported below matches the `contam_removed` reported by `preprocess()` at the gene level.

In [ ]:
# === Contamination filter: barcode-collision visualizations ===
# Reconstruct the cross-run 10X-barcode-collision structure the contamination filter acts on,
# using the same raw gene-level object preprocess() reads. filter_barcode_collisions() is
# already defined above (it is what contamination_keep_mask() calls inside preprocess()), so
# these diagnostics reflect exactly the cells preprocess() drops.
import seaborn as sns
import matplotlib.pyplot as plt

_gene_raw = sc.read_h5ad(
    f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_genes_bc_anndata_mincounts500.h5ad"
)
viz_obs = _gene_raw.obs[["donor", "tissue", "LR_library_id"]].copy()
viz_obs["10X_barcode"] = [str(x).split("-1")[0] for x in _gene_raw.obs_names]   # PacBio-orientation barcode
del _gene_raw
gc.collect()

# per-donor decontamination (same grouping the filter uses) -> kept obs + overlap dicts
per_donor_kept = {}
per_donor_overlaps = {}
for donor in viz_obs["donor"].unique():
    od, dd = filter_barcode_collisions(viz_obs[viz_obs["donor"] == donor], "LR_library_id")
    per_donor_kept[donor] = dd
    if len(od) > 0:
        per_donor_overlaps[donor] = od

# all-donor overlap dict (for the pooled heatmap)
overlap_dict_all, _ = filter_barcode_collisions(viz_obs.copy(), "LR_library_id")

# contamination_filter boolean on the full obs (True = kept, i.e. no within-donor collision)
kept_combined = pd.concat(per_donor_kept.values())
kept_combined["contamination_filter"] = True
contam_obs = viz_obs.join(kept_combined[["contamination_filter"]])
contam_obs["contamination_filter"] = contam_obs["contamination_filter"].fillna(False).astype(bool)
print(contam_obs["contamination_filter"].value_counts())
print(f"Contamination-filtered (dropped) cells: {(~contam_obs['contamination_filter']).sum():,} "
      f"(matches preprocess() contam_removed = {gene_rec['contam_removed']:,.0f})")

In [ ]:
# --- Per-donor barcode-overlap heatmaps ---
sc.settings.set_figure_params(dpi=80, fontsize=5, facecolor="white", frameon=True,
                              figsize=(5, 5), vector_friendly=False)

def _sort_by_number(item):
    digits = "".join(c for c in str(item) if c.isdigit())
    return int(digits) if digits else 0

unique_donors = sorted(per_donor_overlaps.keys(), key=_sort_by_number)
num_donors = len(unique_donors)
if num_donors == 0:
    print("No donors with overlapping barcodes between 10X runs; skipping heatmap.")
else:
    num_cols = min(4, num_donors)
    num_rows = (num_donors + num_cols - 1) // num_cols
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(7.5 * num_cols, 7 * num_rows), squeeze=False)
    for idx, donor in enumerate(unique_donors):
        row, col = idx // num_cols, idx % num_cols
        overlap_df = pd.DataFrame(pd.Series(per_donor_overlaps[donor]))
        overlap_df.index.names = ["10X Run 1", "10X Run 2"]
        overlap_df.columns = ["Overlap (%)"]
        overlap_matrix = overlap_df["Overlap (%)"].unstack().fillna(0)
        sns.set_style("ticks")
        sns.heatmap(overlap_matrix, ax=axes[row, col], linewidths=0.05, cmap="Blues", linecolor="black")
        axes[row, col].set_title(f"Donor: {donor}")
        axes[row, col].set_xlabel("10X Run 2")
        axes[row, col].set_ylabel("10X Run 1")
    for idx in range(num_donors, num_rows * num_cols):
        axes[idx // num_cols, idx % num_cols].axis("off")
    plt.savefig(figpath("01_08_contamination_heatmaps_duplicates_removed.pdf"))
    plt.show()

In [ ]:
# --- Barcode collisions across all 10X runs (all donors) ---
if len(overlap_dict_all) == 0:
    print("No overlapping barcodes between any 10X runs; skipping heatmap.")
else:
    overlap_df_all = pd.DataFrame(pd.Series(overlap_dict_all))
    overlap_df_all.index.names = ["10X Run 1", "10X Run 2"]
    overlap_df_all.columns = ["Overlap (%)"]
    overlap_matrix_all = overlap_df_all["Overlap (%)"].unstack().fillna(0)
    sc.settings.set_figure_params(dpi=80, fontsize=10, facecolor="white", frameon=True,
                                  figsize=(5, 5), vector_friendly=False)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.set_style("ticks")
    sns.heatmap(overlap_matrix_all, ax=ax, linewidths=0.05, cmap="Blues", linecolor="black")
    ax.set_title("Barcode collisions across all 10X runs (all donors)")
    ax.set_xlabel("10X Run 2")
    ax.set_ylabel("10X Run 1")
    plt.savefig(figpath("01_08_barcode_collisions_all_runs.pdf"))
    plt.show()

In [ ]:
# --- Per-run overlapping-barcode counts: scatter (overlap vs total cells) ---
temp = (
    contam_obs.groupby(["LR_library_id", "contamination_filter"], observed=False)
    .size().reset_index(name="contamination_counts")
)
temp = temp.join(
    temp.groupby(["LR_library_id"], observed=False)["contamination_counts"]
    .sum().reset_index(name="total_counts").set_index("LR_library_id"),
    on="LR_library_id",
)
temp = temp[temp["contamination_filter"] == False]
temp = temp[temp["total_counts"] > 0]
temp["LR_library_id"] = temp["LR_library_id"].astype(str)

sc.settings.set_figure_params(dpi=80, dpi_save=300, fontsize=6, facecolor="white", frameon=True,
                              figsize=(2.0, 2.0), vector_friendly=False, transparent=True, format="pdf")
sns.set_style("ticks", rc={"text.color": "black", "linecolor": "black", "axes.edgecolor": "black",
                           "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black"})
fig, ax = plt.subplots(figsize=(6.0, 3.0))
sns.scatterplot(temp, x="total_counts", y="contamination_counts", hue="LR_library_id", palette="tab20", ax=ax)
plt.legend(ncol=1, handletextpad=0.2, markerscale=0.6, labelspacing=0.0, frameon=False,
           borderpad=0.5, columnspacing=0.5, borderaxespad=0.1, loc=6, bbox_to_anchor=(1.0, 0.5),
           title="Run ID", alignment="left", title_fontproperties={"weight": "bold"})
ax.tick_params(axis="both", which="major", pad=0, size=2)
ax.set_xlabel("Total cell count", labelpad=0.5)
ax.set_ylabel("Number of overlapping barcodes", labelpad=1.0)
plt.savefig(figpath("01_08_overlap_duplicates_removed.pdf"))
plt.show()

In [ ]:
# --- Per-run overlapping-barcode counts: ranked bar ---
temp_sorted = temp.sort_values("contamination_counts", ascending=False)
fig, ax = plt.subplots(figsize=(8.0, 4.0))
sns.barplot(temp_sorted, y="contamination_counts", x="LR_library_id",
            order=temp_sorted["LR_library_id"], ax=ax)
ax.set_xlabel("Run ID")
ax.set_ylabel("Number of overlapping barcodes")
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")
plt.savefig(figpath("01_08_overlapping_barcodes_per_run.pdf"))
plt.show()

In [ ]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import os, gc
import numpy as np
import pandas as pd
import scanpy as sc
gc.enable()

BASEDIR = "./../pacbio"
H5AD_DIR = f"{BASEDIR}/h5ads/"

LEVELS = ["genes", "pbids", "ensemblids", "ensemblids_annotatedonly"]
# LEVELS = [ "pbids", "ensemblids", "ensemblids_annotatedonly"]

# reload figure_paths: a live kernel caches FIG_MAP
import importlib
import figure_paths
importlib.reload(figure_paths)
from figure_paths import figpath  # routes figures/ -> figures/figureN/ via FIG_MAP


In [18]:
adata = sc.read_h5ad(
    f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_pbids_bc_anndata_preprocessed.h5ad"
)
adata

AnnData object with n_obs × n_vars = 203311 × 854410
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'tube_id', 'n_counts', 'donor', 'tissue', 'tube_label', 'LR_library_id', 'SR_sample_id', 'popv_prediction', 'popv_prediction_score', 'cell_ontology_class', 'broad_cell_class'
    var: 'gene_ids', 'molecule_id', 'gene_name', 'transcript_id', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [10]:
adata = sc.read_h5ad(
    f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_ensemblids_bc_anndata_preprocessed.h5ad"
)
adata

AnnData object with n_obs × n_vars = 203311 × 489842
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'tube_id', 'n_counts', 'donor', 'tissue', 'tube_label', 'LR_library_id', 'SR_sample_id', 'popv_prediction', 'popv_prediction_score', 'cell_ontology_class', 'broad_cell_class'
    var: 'transcript_id', 'gene_name', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [12]:
# === Feature metadata export for the DEPOSITED pbid-level object (figshare) ===
#
# The classification CSVs LOOK like "feature x sample" -- they carry a
# `tissue_source` column and ~11.5M rows against an atlas of 854,410 features --
# but they are not. Verified 2026-09-01 on both files:
#
#     11,529,975 rows  carrying  11,529,975 DISTINCT isoform ids
#
# Every PB structure appears exactly once, because the global re-collapse
# (syncronize.smk) makes ids unique atlas-wide. `tissue_source` is therefore a
# per-feature LABEL (the sample that contributed that structure), never a
# grouping key, and feature annotation cannot differ by sample. The extra rows
# are simply the pre-filter structures that 01_08 removed above.
#
# So this export is a PURE FILTER: no aggregation, no de-duplication, one row
# out per deposited feature. Coverage was checked for both files -- all 854,410
# atlas features present, 0 missing -- and the assertion below re-checks it.
#
# Only the pbid level is exported this way. The ensemblid classification joins
# on a SQANTI3 `transcript_id`, while the h5ad's `transcript_id` comes from
# pigeon (01_03); the two disagree for 19,502 of 489,842 features (4.0%). An
# ensemblid table must therefore be derived from the pbid table through the
# h5ad's own var mapping, not by joining that CSV directly.
#
# Chunked read with dtype=str / keep_default_na=False so values round-trip
# verbatim: with per-chunk dtype inference an int column could serialise as
# "1" in one chunk and "1.0" in the next.

CSV_DIR = "./../csvs"
PBID_H5AD = f"{H5AD_DIR}/all_samples_pacbio_recollapsed_raw_counts_pbids_bc_anndata_preprocessed.h5ad"
CHUNK = 1_000_000

# ORF_length is the CDS GENOMIC SPAN, not the ORF length: it equals
# CDS_genomic_end - CDS_genomic_start for 100% of 254,232 coding rows, so every
# multi-exon ORF is inflated by its intron content and 88.7% exceed their own
# transcript length. Renamed rather than dropped, so nothing is silently lost
# and no one downstream reads it as a protein length. Use CDS_length instead.
# (Empty in the pigeon file, which produces no ORF/CDS/NMD calls at all.)
RENAME_BAD_ORF = True

CLASSIFICATIONS = {
    "pigeon":  f"{CSV_DIR}/all_samples_recollapsed_pigeon_classification.csv",
    "sqanti3": f"{CSV_DIR}/all_samples_recollapsed_sqanti3_classification.csv",
}

_a = sc.read_h5ad(PBID_H5AD, backed="r")
keep = set(_a.var_names)
_a.file.close()
del _a
print(f"deposited pbid features: {len(keep):,}\n")

for tag, src in CLASSIFICATIONS.items():
    out = f"{CSV_DIR}/all_samples_pbids_feature_metadata_{tag}.csv"
    n_in = n_out = 0
    first = True
    for chunk in pd.read_csv(src, chunksize=CHUNK, dtype=str, keep_default_na=False):
        n_in += len(chunk)
        sub = chunk[chunk["isoform"].isin(keep)]
        if RENAME_BAD_ORF and "ORF_length" in sub.columns:
            sub = sub.rename(columns={"ORF_length": "ORF_genomic_span"})
        n_out += len(sub)
        sub.to_csv(out, mode="w" if first else "a", header=first, index=False)
        first = False

    assert n_out == len(keep), (
        f"{tag}: wrote {n_out:,} rows for {len(keep):,} deposited features. "
        "The classification no longer covers the atlas one-to-one -- do NOT "
        "publish this file until the mismatch is understood.")
    print(f"[{tag}] {n_in:,} rows -> {n_out:,} (one per deposited feature) "
          f"| {os.path.getsize(out) / 2**20:,.0f} MB | saved {out}")

print("\nJoin key: `isoform` <-> adata.var_names of the deposited pbid object.")

deposited pbid features: 854,410

[pigeon] 11,529,975 rows -> 854,410 (one per deposited feature) | 297 MB | saved ./../csvs/all_samples_pbids_feature_metadata_pigeon.csv
[sqanti3] 11,529,975 rows -> 854,410 (one per deposited feature) | 256 MB | saved ./../csvs/all_samples_pbids_feature_metadata_sqanti3.csv

Join key: `isoform` <-> adata.var_names of the deposited pbid object.


In [17]:
df = pd.read_csv("./../csvs/all_samples_pbids_feature_metadata_pigeon.csv")
df.head()

,isoform,chrom,strand,length,exons,structural_category,associated_gene,associated_transcript,ref_length,ref_exons,...,dist_to_polyA_site,within_polyA_site,polyA_motif,polyA_dist,polyA_motif_found,ORF_seq,ratio_TSS,fl_assoc,cell_barcodes,tissue_source
0,PB.1.929,chr1,+,736,2,novel_not_in_catalog,ENSG00000286448,novel,736.0,2.0,...,NaN,NaN,AATAGA,-8.0,NaN,NaN,NaN,2,"612,613",T10
1,PB.1.1951,chr1,+,342,1,full-splice_match,MTND1P23,ENST00000416931.1,372.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,298,"1124,1125,1126,869,407,1127,1128,1129,1130,113...",T10
2,PB.1.2447,chr1,+,433,1,full-splice_match,MTCO1P12,ENST00000414273.1,1543.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,497,"1453,1454,1455,1456,177,1457,1458,1459,1053,14...",T10
3,PB.1.3427,chr1,+,692,1,full-splice_match,MTCO2P12,ENST00000427426.1,682.0,1.0,...,NaN,NaN,ATTAAA,-36.0,NaN,NaN,NaN,4,"1768,1769,1770,1114",T10
4,PB.1.3464,chr1,+,80,1,full-splice_match,MTATP8P1,ENST00000467115.1,207.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49,"402,731,1772,1773,1774,1775,1776,1777,1778,177...",T10


In [19]:
df = pd.read_csv("./../csvs/all_samples_pbids_feature_metadata_sqanti3.csv")
df.head()

,isoform,chrom,start,end,strand,length,exons,structural_category,subcategory,FSM_class,...,dist_to_polyA_site,within_polyA_site,polyA_motif,polyA_dist,polyA_motif_found,ratio_TSS,FL.count_fl,FL.fl_assoc,filter_result,tissue_source
0,PB.1.929,chr1,266812,268649,+,736,2,novel_not_in_catalog,at_least_one_novel_splicesite,A,...,NaN,False,AATAGA,-8.0,True,NaN,2,2,Isoform,T10
1,PB.1.1951,chr1,629089,630004,+,775,2,genic,multi-exon,A,...,NaN,False,NaN,NaN,False,NaN,354,395,Artifact,T10
2,PB.1.2447,chr1,630750,632686,+,1081,4,genic,multi-exon,A,...,NaN,False,NaN,NaN,False,NaN,655,676,Artifact,T10
3,PB.1.3427,chr1,632966,633658,+,692,1,full-splice_match,mono-exon,A,...,NaN,False,ATTAAA,-36.0,True,NaN,4,4,Artifact,T10
4,PB.1.3464,chr1,633740,634228,+,416,2,genic,multi-exon,A,...,NaN,False,NaN,NaN,False,NaN,43,50,Artifact,T10


In [20]:
df["filter_result"].value_counts()

filter_result
Isoform     691933
Artifact    162477
Name: count, dtype: int64

In [21]:
df[df["filter_result"] == "Isoform"]["structural_category"].value_counts()

structural_category
incomplete-splice_match    326756
novel_not_in_catalog       161740
novel_in_catalog           110380
full-splice_match           83571
fusion                       6127
genic                        3189
antisense                      71
genic_intron                   56
intergenic                     43
Name: count, dtype: int64